# **Frugal AI: Rethinking model design and objective for a sustainable future**

**PSL-Week: Practical work notebook for part 2**

This document presents instructions regarding the second practical work session of the Frugal-AI PSL-Week. All the materials (slides, notebooks, base codes) can be recovered from the corresponding [GitHub repository](https://github.com/Deyht/frugal_ai).

The notebook is expected to be run on a Google Colab environment with a T4 GPU:
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Deyht/frugal_ai/blob/main/frugal_ai_opt_ann_correction.ipynb)


This notebook is **not intended as a standalone** document; the slides should be used as a reference to understand several of the explanations provided in the analysis of the results.

# Energy budget based CNN base optimization on GPU

This part addresses the optimization of Artificial Neural Network models for numerical efficiency and, therefore, energy efficiency.

For this exercise, we will use the ASIRRA (Animal Species Image Recognition for Restricting Access) dataset, which comprises 25000 labeled images of Cats and Dogs. The objective is to explore network structures, following the lecture's guidelines, to build a classification model that **maximizes classification accuracy under a given energy budget**. As discussed in the slides, both training and inference contribute to the model's environmental budget. To simplify the problem so it is compatible with a practical work session, we will assume the model's training cost is negligible and account only for electricity consumption during inference (which would match a long-lasting, broadly distributed model). We will also not factor in any infrastructure cost. Those are very strong approximations, and we invite the participants to refer back to the slide for a more complete view of the impact of ITC systems. At the end, **for our practical case, we consider that the energy consumed is simply proportional to the execution time**.

This practical work will be carried out through Google Colab, granting each participant access to an Nvidia T4 GPU (Turing, 2560 CUDA cores, 16 GB RAM, peak FP32: 8.141 TFLOPS, and FP16: 65.13 TFLOPS, 70W peak consumption). Our energy budget will be based on the compute time required to process a set of 2048 test images, with a maximum budget of 300 ms. Your objective will be to design a model that achieves the highest possible classification accuracy (maximizing usefulness) while remaining under this budget. Note that a model reaching only 80% accuracy on this task (1 error out of 5 predictions) can barely be considered useful.

The default version of this notebook handles the data loading, model training, and inference for a simple LeNET-5-like architecture. The ANN model is built and trained using the custom [CIANNA](https://github.com/Deyht/CIANNA) framework. You should be able to adapt the architecture with only minor adjustments to the appropriate notebook cells, taking inspiration from the provided example. You can refer to CIANNA's WIKI page for a complete framework description and access to the full list of supported layers and their configurations. You can also look at some example scripts provided in CIANNA's GitHub repository. Of course, you are also free to replace these parts with any other Deep Learning development framework you are familiar with (TensorFlow, PyTorch, etc.), as long as you correctly measure the inference time using a T4 GPU over Google Colab.

### **CIANNA installation**

This notebook uses CIANNA to build and train network architectures. CIANNA is a general-purpose artificial neural network development framework (like TensorFlow or PyTorch) designed mainly for astronomical applications. Still, it is perfectly usable for classical CNN implementations. It is very fast for small networks, and it provides several performance measurement tools that will be useful in the context of this exercise.

CIANNA is coded in C and CUDA for GPU acceleration and is controlled through a Python interface. As it is not available in Colab by default, we start by installing it.

**Link to the CIANNA github repository**
https://github.com/Deyht/CIANNA

In [ ]:
%%bash

cd /content/
git clone https://github.com/Deyht/CIANNA -b experimental

cd CIANNA
cmake -B build -D USE_BLAS=off
cmake --build build
cd ..

### **ASIRRA**

The ASIRRA (Animal Species Image Recognition for Restricting Access) dataset was originally used for CAPTCHA and HIP (Human Interactive Proofs).

The original dataset comprises 25000 images of variable resolution (averaging around 350x500) and is evenly distributed over the two classes "Cat" and "Dog". For this exercise, we provide two reduced versions in the form of padded and resized RGB images at either 128x128 or 256x256 as two binary files. This construction is necessary so the dataset can fit into the limited amount of RAM available through the free tier of Colab. You can download one or both sets to test the impact of the input resolution on your network. In these files, the first 12500 images are of Cats, and the next 12500 are of Dogs. The last 1024 images of each class will be excluded to form our reference test dataset (total size 2048).

#### Downloading and visualizing the data

We start by downloading and visualizing the raw data. You can get one or both of the resized versions.

In [ ]:
%%bash

cd /content

wget https://share.obspm.fr/s/6TBsCpAASeETH3S/download/asirra_bin_128.tar.gz
tar -xvzf asirra_bin_128.tar.gz

#wget https://share.obspm.fr/s/52nxyfn7PjzawSe/download/asirra_bin_256.tar.gz
#tar -xvzf asirra_bin_256.tar.gz

#### Import and global settings for the noebook (rerun if the kernel crashes)

In [ ]:
!pip install wurlitzer # For proper display of CIANNA outputs in a notebook
%load_ext wurlitzer

import numpy as np
import matplotlib.pyplot as plt
import os, sys, gc, glob, time, cv2
from threading import Thread
from PIL import Image
import albumentations as A

sys.path.insert(0,glob.glob('/content/CIANNA/build/')[-1]) #if compiled with cmake
import CIANNA as cnn

#### Data handling and augmentation

To ease data manipulation and hyperparameter exploration, we first provide a set of helper functions.  
**Every time you want to change the content of these functions, you will need to rerun this cell.**

In [ ]:

def data_prep(nb_images_per_iter, raw_image_size, image_size, test_mode=0):
  #Data arrays are declared as global so we can work in place to reduce RAM footprint
  global raw_images, input_data, targets, input_val, targets_val

  raw_images = np.reshape(np.fromfile("asirra_bin_%d.dat"%(raw_image_size), dtype="uint8"), (25000, raw_image_size, raw_image_size,3))

  if(test_mode == 0):
    input_data = np.zeros((nb_images_per_iter,3*image_size**2), dtype="float32") #CIANNA expects "float32" arrays
    targets = np.zeros((nb_images_per_iter,2), dtype="float32")

  input_val = np.zeros((2048,3*image_size**2), dtype="float32")
  targets_val = np.zeros((2048,2), dtype="float32")


def create_augmented_batch(A_transform):

  nb_images = np.shape(input_data)[0]

  for i in range(0,nb_images):

    l_class = np.random.randint(0,2)
    l_id = np.random.randint(0,12500 - 1024)
    #Last 1024 images of each class kept only for the val/test set

    patch = raw_images[l_class*12500+l_id]
    transformed = A_transform(image=patch)
    patch_aug = transformed['image']

    l_image_size = np.shape(patch_aug)[0]

    #CIANNA expects data formated as 2D numpy arrays representing a list of flattened images (with every channel flattened after the others)
    for depth in range(0,3): #We normalize based on mean pixel value
      input_data[i,depth*l_image_size**2:(depth+1)*l_image_size**2] = (patch_aug[:,:,depth].flatten("C") - 100.0)/155.0

    targets[i,:] = 0.05 #Label smoothing during training
    targets[i,l_class] = 0.95

  return input_data, targets


def create_validation_set(A_transform):

  for i in range(0,2048):

    l_class = i // 1024

    patch = raw_images[(1+l_class)*(12500 - 1024) + i]
    transformed = A_transform(image=patch)
    patch_aug = transformed['image']

    l_image_size = np.shape(patch_aug)[0]

    for depth in range(0,3):
      input_val[i,depth*l_image_size**2:(depth+1)*l_image_size**2] = (patch_aug[:,:,depth].flatten("C") - 100.0)/155.0

    targets_val[i,:] = 0.0
    targets_val[i,l_class] = 1.0

  return input_val, targets_val


def free_data_helper():
  global raw_images, input_data, targets, input_val, targets_val
  if 'raw_images' in globals():
    del (raw_images)
  if 'input_data' in globals():
    del (input_data)
  if 'targets' in globals():
    del (targets)
  if 'input_val' in globals():
    del (input_val)
  if 'targets_val' in globals():
    del (targets_val)
  gc.collect()
  return

We can test the helper functions with a simple example and display the produced images. The next cell illustrates how we can create an augmented batch of images for training from the raw image dataset.  

Use this example to test the effect of combining different transform operations for data augmentation. You can also test the impact of the image resolution and of the position of the resize transformation in the augmentation list.

In [ ]:
%load_ext wurlitzer

class_text = ["cat","dog"]
raw_image_size = 128
image_size = 64

v_width = 8; v_height = 5
nb_images_per_iter = v_width*v_height

#See Albumentation documentation for a list of existing augmentations
train_transform = A.Compose([
  #A.Affine(scale=(...,...), keep_ratio=False, rotate=(...,...), interpolation=1),
  A.Resize(height=image_size, width=image_size, interpolation=1, p=1.0),
  A.ColorJitter(brightness=(0.9,1.1), contrast=(0.9,1.1), saturation=(0.9,1.1), hue=0.05, p=1.0),
  A.ToGray(p=0.02),
  A.HorizontalFlip(p=0.5),
])

val_transform = A.Compose([
    A.Resize(height=image_size, width=image_size, interpolation=1),
])

data_prep(nb_images_per_iter, raw_image_size, image_size)
input_data, targets = create_augmented_batch(train_transform)

fig, ax = plt.subplots(v_height, v_width, figsize=(v_width*1.5,v_height*1.5), dpi=200, constrained_layout=True)
patch = np.zeros((image_size,image_size,3), dtype="uint8")

for i in range(0, v_width*v_height):
  c_x = i // v_width; c_y = i % v_width
  #Images in the augmented input_data array are directly in the CIANNA format.
  #We need to convert them back to classical RGB for display.
  for depth in range(0,3):
    patch[:,:,depth] = np.reshape(input_data[i,depth*image_size**2:(depth+1)*image_size**2]*155 + 100,(image_size,image_size))

  ax[c_x,c_y].imshow(patch)
  ax[c_x,c_y].text(4, 10, class_text[(np.argmax(targets[i]))], c="red", fontsize=10, clip_on=True)
  ax[c_x,c_y].axis('off')

plt.show()

free_data_helper()

#### Training a network

The original example uses a LeNet-5-inspired network. The training cell below runs our sixth configuration.

As stated at the beginning of the notebook, **your objective is to find an architecture that maximizes classification accuracy, while keeping the inference time under 300 ms**.

The only limiting rules are not to use external data or pretrained models on another dataset, not to change the data split between training and validation/test, and to run inference on a T4 GPU on Colab to measure inference time.
Other than that, you are free to:
 * Change the input image resolution
 * Modify/expand the augmentation policy
 * Modify/expand the network backbone

To stimulate your search, we provide a shared [Google Sheet](https://docs.google.com/spreadsheets/d/1l3l5zfHfAsi5AF4IU4P-5OQs0nl4cRzQpoHdWkRBqUc/edit?usp=sharing) to be used as a dynamic leaderboard. You can tackle this part individually, in pairs, or by forming teams. Once you have a working model, you can enter its properties/results in the leaderboard. It will only be accessible for the duration of this course.

**Some strategies to help you explore model architectures:**
 * 1) Try to add layers in the network backbone and evaluate their impact on accuracy and compute time.
 * 2) Use the provided table of performance per layer to identify bottlenecks in your architecture (Remember that a layer's performance is not only dictated by its own parameters but also by the shape of its input, which depends on the configuration of the previous layer).
 * 3) When increasing the number of parameters in your architecture (number of layers or their size), you will be more prone to overtraining (test set error increasing). For this reason, changes to the architecture should be accompanied by changes to your image augmentation policy. Overtraining can be monitored by following the "fwd mean loss" reported by CIANNA, which is computed on the validation dataset (identical to the test dataset in our case). The validation error as a function of the epoch is also reported in the *error.txt* file that is created automatically by CIANNA.
 * 4) Increasing the image size is a good way to improve accuracy, but there is a diminishing return at some point. However, this will have a strong negative impact on your computing time. Try to reduce the activation maps' resolution more aggressively over a few layers when using higher input resolutions. Keep an eye on the receptive field of the convolutional part of your backbone.

**Points to keep an eye on:**
 * The size of the last activation maps before your first dense layer will have a strong impact on the number of parameters in your model, but it will not necessarily correlate with its accuracy. Keep an eye on the size of your activation maps before the first dense layer. You could also try to build a fully convolutional architecture to get rid of dense layers. This also has the advantage of making your trained model "compatible" with many input image resolutions. Again, the receptive field will be very important with such architecture. Examples are available in the CIANNA repository (see the ImageNET network backbone).
 * The batch size has a direct effect on the compute speed of your model, with larger values allowing faster computing (up to some limit). However, you are limited by the amount of GPU memory used by your model. Also, in this exercise, speed only matters at inference time. In contrast, small batch sizes are usually preferable during training to achieve higher accuracy and converge in fewer training steps.
 * Using mixed precision is also an efficient way to increase computing speed, but it can reduce the accuracy (usually by only a small amount). Unless you choose to use a very deep architecture, you will likely achieve better results with FP16C_FP32A activated for both training and inference.
 * Finally, use existing knowledge available regarding architecture design. You are free to be inspired by any architecture you know or find. You will find architecture examples in the CIANNA repository, but you are free to explore various architecture designs (AlexNET, VGG, DenseNET, YOLO/darknet, etc), as long as the required layers are available in CIANNA.

*Link to the [CIANNA](https://github.com/Deyht/CIANNA) repository. You can refer to CIANNA's [website](https://cianna.obspm.fr/) for a complete framework description. You can also look at the full [API documentation](https://cianna.obspm.fr/content/API/API.html)-Interface-API-documentation) to add layer types that are absent from the LeNET-5 example.
The saved models are available in the "net_save" repository that is automatically created when starting network training. The default naming scheme only refers to the training iteration, so rename your saving files with comprehensive information about your model to keep track of your progress. A saved model can be uploaded to a new Colab session for inference or further training.*

## Comparaison des configurations 1 à 6

Chaque ligne décrit une expérience. Dans la colonne « convolutions », chaque nombre donne le nombre de filtres d’une couche et la barre oblique sépare deux résolutions. Les temps portent sur les **2 048 mêmes images**. Le budget est de **5 s sur le CPU de la salle** et de **300 ms sur le T4**.

| Config | Machine · entrée | Convolutions par résolution | Tête de classification | Précision entraînement → inférence | Accuracy et checkpoint | Inférence |
|---|---|---|---|---|---|---:|
| 1 | CPU · 64 × 64 | 8 / 16 | Dense 256 → 128 → 2, dropout 80 % et 50 % | FP32 → FP32 | 83,89 % à 500 | 0,70 s |
| 2 | CPU · 64 × 64 | 16, 16 / 32, 32 / 64 | Dense 256 → 128 → 2, dropout 50 % et 30 % | FP32 effectif → FP32 | 92,92 % à 500 | 3,63 s |
| 3 · `v3_gap` | CPU · 64 × 64 | 16 / 24, 24 / 48, 48 / 96, 96 | Conv 1 × 1 à 2 filtres + moyenne globale | FP32 → FP32 | 90,48 % au checkpoint 100, entraînement interrompu | 2,70 s au checkpoint 100 |
| 4 | T4 · 96 × 96 | 24 / 48, 48 / 96, 96 / 128 | Conv 1 × 1 à 2 filtres + moyenne globale | Mixte → mixte | 89,70 % à 150 en validation ; 50,00 % à 500 | 154,5 ms au checkpoint 500 |
| 5 | T4 · 96 × 96 | Identiques à la 4 | Identique à la 4 | **FP32 → mixte** | **94,29 % à 500** | **136,1 ms** |
| 6 | T4 · **128 × 128** | Identiques à la 5 | Identique à la 5 | FP32 → mixte | À mesurer | À mesurer |

**Réglages d’entraînement.** CIANNA, RMSProp avec momentum 0,9, batch 32 et 500 itérations prévues. Les configs CPU présentent 4 096 images par itération, les configs T4 en présentent 2 048. Le batch d’inférence vaut respectivement 32 et 128. « Mixte » signifie calcul FP16 avec accumulation FP32. La config 1 utilise un taux d’apprentissage maximal de 0,0004, puis les configs 2 à 6 utilisent 0,0003. Dès la config 2, on élargit les variations de couleur et on ajoute de petites rotations et variations d’échelle. La config 5 a pris **15,09 min**. C’est la première dont on a chronométré la boucle d’entraînement complète.

### Pourquoi chaque config ?

1. **Fixer une référence.** Deux convolutions et une tête dense donnent rapidement un premier score. L’inférence ne prend que 0,70 s sur CPU, ce qui laisse de la marge pour essayer un réseau plus profond.
2. **Mieux exploiter les images de 64 × 64.** On ajoute des convolutions, on réduit le dropout et on renforce les augmentations. Ensemble, ces changements font gagner **9,03 points** tout en restant sous 5 s. Cet essai ne permet pas d’attribuer le gain à une seule modification.
3. **Tester une tête sans couche dense.** La deuxième convolution à pleine résolution prenait environ 42 % du temps des couches dans la config 2. On avance le premier pooling et on ajoute des convolutions sur les petites cartes. La moyenne globale transforme les deux cartes finales en scores chat et chien. À l’itération 100, la config 3 atteint **90,48 %**, contre **87,21 %** pour la config 2 au même stade. On ignore où elle aurait terminé.
4. **Tester davantage de détails sur le T4.** On passe à 96 × 96. Une convolution 4 × 4 de stride 2 ramène immédiatement la carte à 48 × 48 pour limiter le coût. Le modèle atteint 89,70 % à 150, puis les pertes d’entraînement et de validation augmentent. Dès 400, il prédit « chat » pour toutes les images. Le temps de 154,5 ms correspond au modèle évalué à 500.
5. **Tester la stabilité de l’entraînement FP32.** On reprend l’architecture de la 4 et on garde l’inférence en précision mixte. On conserve aussi le meilleur checkpoint observé tous les 50 passages de la boucle. Cette fois, le meilleur est le dernier, à **94,29 %**. C’est le meilleur résultat mesuré sur T4 sous 300 ms.
6. **Tester la résolution d’origine des données.** La config 5 dispose de près de 164 ms de marge. On utilise donc les images ASIRRA en 128 × 128 au lieu de les réduire à 96 × 96. L’architecture et les réglages de la 5 sont conservés. L’essai sera concluant si le même checkpoint dépasse **94,29 %** et reste sous **300 ms**.

### Config 6 · Entraînement

Dans la session Colab déjà connectée au T4, exécuter les deux cellules de code suivantes dans l’ordre. Elles réutilisent les données et les fonctions de préparation déjà chargées.

In [ ]:
import torch
import csv
from pathlib import Path
from tempfile import mkdtemp

raw_image_size = 128
image_size = 128  # Config 6 : exploiter toute la résolution des images ASIRRA.
eval_batch = 128
precision = "FP16C_FP32A"
train_precision = "FP32C_FP32A"
val_transform = A.Compose([A.Resize(image_size, image_size, interpolation=1)])

def build_model():
  relu = cnn.relu(leaking=0.05, saturation=64.0)
  cnn.input(spatial_shape=[image_size, image_size, 1], nb_features=3)
  cnn.conv(f_size=(4,4), stride=(2,2), padding=(1,1), nb_filters=24, activation=relu)
  for width in (48, 96):
    cnn.pool(p_size=(2,2), p_type="MAX")
    for _ in range(2):
      cnn.conv(f_size=(3,3), padding=(1,1), nb_filters=width, activation=relu)
  cnn.pool(p_size=(2,2), p_type="MAX")
  cnn.conv(f_size=(3,3), padding=(1,1), nb_filters=128, activation=relu)
  cnn.conv(f_size=(1,1), nb_filters=2, activation="LIN")
  cnn.pool(p_size=(1,1), p_global=1, p_type="AVG", activation="SMAX")

def check_budget():
  cnn.init(b_size=eval_batch, inference_only=1, dynamic_load=1,
           comp_meth="C_CUDA", mixed_precision=precision)
  try:
    build_model()
    data_prep(0, raw_image_size, image_size, test_mode=1)
    x, y = create_validation_set(val_transform)
    cnn.create_dataset("TEST", len(y), x, y)
    cnn.forward(drop_mode="AVG_MODEL", no_error=1, saving=0, return_output=0)
    torch.cuda.synchronize()
    times = []
    for _ in range(5):
      start = time.perf_counter()
      cnn.forward(drop_mode="AVG_MODEL", no_error=1, saving=0, return_output=0)
      torch.cuda.synchronize()
      times.append(1000 * (time.perf_counter() - start))
    median = float(np.median(times))
    print(f"Batch {eval_batch}, 2 048 images : {np.round(times, 1)} ms ; médiane {median:.1f} ms")
    return median
  finally:
    free_data_helper()

latency_ms = check_budget()

if latency_ms >= 300:
  raise RuntimeError("Budget dépassé. Ajuster eval_batch et relancer la cellule de chronométrage.")
nb_images_per_iter = 2048
load_iter = 0
train_transform = A.Compose([
  A.Resize(image_size, image_size, interpolation=1),
  A.ColorJitter(brightness=(0.85,1.15), contrast=(0.85,1.15), saturation=(0.85,1.15), hue=0.05, p=0.8),
  A.ToGray(p=0.02), A.HorizontalFlip(p=0.5),
  A.Affine(scale=(0.9,1.1), rotate=(-10,10), interpolation=1, p=0.3),
])

#This funtion allow to launch data augmentation on a separate thread.
#This way we can train on the GPU and generate new agumented examples in parallel.
def data_augm():
  input_data, targets = create_augmented_batch(train_transform)
  cnn.delete_dataset("TRAIN_buf", silent=1)
  cnn.create_dataset("TRAIN_buf", nb_images_per_iter, input_data[:,:], targets[:,:], silent=1)
  return

cnn.init(b_size=32, optimizer=cnn.rms_prop(momentum=0.9), wema=0, dynamic_load=1,
  comp_meth='C_CUDA', mixed_precision=train_precision)
build_model()

data_prep(nb_images_per_iter, raw_image_size, image_size)

input_val, targets_val = create_validation_set(val_transform)
cnn.create_dataset("VALID", 2048, input_val, targets_val)
# forward() lit TEST ; les deux noms utilisent le même split officiel du TP.
cnn.create_dataset("TEST", 2048, input_val, targets_val)


#Create fist augmentation before parallelization
input_data, targets = create_augmented_batch(train_transform)
cnn.create_dataset("TRAIN", nb_images_per_iter, input_data[:,:], targets[:,:])

warmup = 10
cycle_length = 500
nb_cycles = 1
#Cycle length and/or the number of cycle should be increased with the complexity of the network and task

l_r_max = 3e-4
l_r_min = 3e-6

#In case the creation of new augmented data is too long compared to training, you can
#increase the number of training iteration over a single augmentation
nb_iter_per_augm = 1
if(nb_iter_per_augm > 1):
  shuffle_frequency = 1
else:
  shuffle_frequency = 0


run_dir = Path(mkdtemp(prefix="config6_t4_", dir="/content"))
checkpoint = run_dir / "config6_best.dat"
log_path = run_dir / "training.csv"
best_accuracy, best_iter = -1.0, 0
# Durée murale de la boucle, augmentations, validations et sauvegardes incluses.
torch.cuda.synchronize()
train_start = time.perf_counter()
with log_path.open("w", newline="") as log:
  writer = csv.writer(log)
  writer.writerow(["iteration", "images_seen", "training_seconds", "validation_accuracy_pct", "best_iteration"])
  for i in range(load_iter, nb_cycles*cycle_length):

    if((i+1) % nb_iter_per_augm == 0):
      t = Thread(target=data_augm)
      t.start()

    if(i < warmup):
      l_r = l_r_min + (l_r_max - l_r_min) * min(i/warmup,1.0)
    else: # Cyclical cosin learning rate decay with warm restart
      cycle_id = i//cycle_length
      in_cycle_progress = (i%cycle_length)/cycle_length
      l_r = l_r_min + 0.5*(l_r_max*0.8**cycle_id - l_r_min)*(1 + np.cos(np.pi*in_cycle_progress))

    cnn.train(nb_iter=1, learning_rate=l_r, weight_decay=0.01, decoupled_wdecay=1,
      confmat=1, control_interv=50, save_every=0, save_optim_every=0, save_bin=1, TC_scale_factor=1.0, shuffle_every=shuffle_frequency)

    if(i == 0):
      cnn.perf_eval()

    if((i+1) % nb_iter_per_augm == 0):
      t.join()
      cnn.swap_data_buffers("TRAIN")
    if (i + 1) % 50 == 0:
      preds = cnn.forward(drop_mode="AVG_MODEL", no_error=1, saving=0, return_output=1)
      accuracy = float(np.mean(preds[:, :2].argmax(1) == targets_val.argmax(1)))
      if accuracy > best_accuracy:
        best_accuracy, best_iter = accuracy, i + 1
        cnn.save(str(checkpoint), bin=1)
      torch.cuda.synchronize()
      elapsed = time.perf_counter() - train_start
      writer.writerow([i + 1, (i + 1) * nb_images_per_iter, round(elapsed, 2),
                       round(100 * accuracy, 4), best_iter])
      log.flush()
      print(f"Iter {i+1} | {elapsed/60:.2f} min | accuracy {accuracy:.2%} | meilleur checkpoint {best_iter}", flush=True)
print(f"Temps total d’entraînement : {elapsed/60:.2f} min. Log : {log_path}")
free_data_helper()

### Config 6 · Évaluation

Cette cellule recharge le meilleur checkpoint, mesure cinq passages après chauffe, puis télécharge les poids et les mesures dans la même archive.

In [ ]:
import torch

val_transform = A.Compose([
    A.Resize(height=image_size, width=image_size, interpolation=1),
])

cnn.init(b_size=eval_batch, inference_only=1, dynamic_load=1,
  comp_meth='C_CUDA', mixed_precision="FP16C_FP32A")

data_prep(0, raw_image_size, image_size, test_mode=1)

input_test, targets_test = create_validation_set(val_transform)
cnn.create_dataset("TEST", 2048, input_test[:,:], targets_test[:,:])


cnn.input(spatial_shape=[image_size,image_size,1], nb_features=3)
cnn.load(str(checkpoint), best_iter, nb_skip_layers=1, bin=1, silent=1)
pred = cnn.forward(drop_mode="AVG_MODEL", no_error=1, saving=0, return_output=1)
torch.cuda.synchronize()
times = []
for _ in range(5):
  start = time.perf_counter()
  cnn.forward(drop_mode="AVG_MODEL", no_error=1, saving=0, return_output=0)
  torch.cuda.synchronize()
  times.append(1000 * (time.perf_counter() - start))

median_ms = np.median(times)
accuracy = np.mean(pred[:, :2].argmax(axis=1) == targets_test.argmax(axis=1))
print(f"2 048 images : {np.round(times, 1)} ms | médiane : {median_ms:.1f} ms | budget < 300 ms : {median_ms < 300}")
print(f"Accuracy : {100 * accuracy:.2f} %")
free_data_helper()
from google.colab import files
from zipfile import ZipFile

archive = run_dir / "config6_resultats.zip"
with ZipFile(archive, "w") as bundle:
  bundle.write(checkpoint, checkpoint.name)
  bundle.write(log_path, log_path.name)
  bundle.writestr("evaluation.csv",
                  "iteration,image_size,batch,accuracy_pct,median_ms,training_seconds\n"
                  f"{best_iter},{image_size},{eval_batch},{100*accuracy:.4f},{median_ms:.3f},{elapsed:.2f}\n")
files.download(str(archive))
